In [8]:
import time
import pandas as pd
import numpy as np
from prophet import Prophet
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [9]:
data = pd.read_csv("../../../data/raw/kathmandu_full_raw_2023_2024.csv")

data["time"] = pd.to_datetime(data["time"])
data = data.sort_values("time").reset_index(drop=True)

In [10]:
prophet_df = pd.DataFrame({
    "ds": data["time"],
    "y":  data["pm2_5"]
})

split = int(np.ceil(0.8 * len(prophet_df)))
train = prophet_df.iloc[:split]
test  = prophet_df.iloc[split:]

In [11]:
# --- Train ---
train_start = time.time()
model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05,  # controls trend flexibility (default 0.05)
    seasonality_prior_scale=10,    # controls seasonality strength (default 10)
    seasonality_mode='multiplicative'  # try this instead of default 'additive'
)
model.fit(train)
train_time = time.time() - train_start

16:17:52 - cmdstanpy - INFO - Chain [1] start processing
16:18:09 - cmdstanpy - INFO - Chain [1] done processing


In [12]:
# --- Predict on train ---
train_forecast = model.predict(train[["ds"]])
train_preds    = train_forecast["yhat"].values

# --- Predict on test ---
inference_start = time.time()
test_forecast = model.predict(test[["ds"]])
inference_time = (time.time() - inference_start) / len(test)
test_preds = test_forecast["yhat"].values

In [13]:
# --- Metrics ---
train_rmse = root_mean_squared_error(train["y"], train_preds)
train_mae  = mean_absolute_error(train["y"], train_preds)
train_r2   = r2_score(train["y"], train_preds)

test_rmse = root_mean_squared_error(test["y"], test_preds)
test_mae  = mean_absolute_error(test["y"], test_preds)
test_r2   = r2_score(test["y"], test_preds)

print(f"Train RMSE: {train_rmse:.4f} | MAE: {train_mae:.4f} | R2: {train_r2:.4f}")
print(f"Test  RMSE: {test_rmse:.4f} | MAE: {test_mae:.4f} | R2: {test_r2:.4f}")
print(f"Training Time:  {train_time:.2f}s")
print(f"Inference Time: {inference_time:.6f}s per sample")

Train RMSE: 9.7045 | MAE: 7.5728 | R2: 0.5958
Test  RMSE: 37.5248 | MAE: 24.7523 | R2: -0.5756
Training Time:  19.44s
Inference Time: 0.000372s per sample
